In [0]:
from pyspark.sql.functions import *

In [0]:
max_dh_bronze = (
    spark.read.table('api_football.bronze.matches_raw')
    .agg(max('dh_processing_br').alias('max_dh_bronze'))
    .collect()[0]['max_dh_bronze'] #transforma de dataframe para variável
) #coletando o máximo registro e transformando em uma váriavel para usar como filtro
df_silver = (
    spark.read.table('api_football.silver.matches')
)
df_matches_raw = (
    spark.read.table('api_football.bronze.matches_raw').alias('mr')
    .join(df_silver.alias('s'), col('s.match_id') == col('mr.match_id'), 'leftanti') # join para pegar os dados que não estão no silver
    .filter(col('mr.dh_processing_br') == max_dh_bronze)
    .filter(col('mr.match_status') == 'Finished')
)

# display(df_matches_raw.count())


In [0]:
df_matches_clean = (
    df_matches_raw
    .filter(col('match_status') == 'Finished') #partidas finalizadas
    .select(
        col('match_id').cast('int'),
        col('match_date'),
        col('match_hometeam_id').cast('int'),
        col('match_hometeam_name'),
        col('match_awayteam_id').cast('int'),
        col('match_awayteam_name'),
        col('match_hometeam_score').cast('int'),
        col('match_awayteam_score').cast('int'),
        when(col('match_hometeam_score') == col('match_awayteam_score'), 'draw')
        .when(col('match_hometeam_score').cast('int') > col('match_awayteam_score').cast('int'), 'home')
        .when(col('match_hometeam_score').cast('int') < col('match_awayteam_score').cast('int'), 'away')
        .alias('match_result'),
        col('match_round').cast('int'),
        col('match_stadium'),
        col('match_referee'),
        col('league_name'),
        col('dh_processing_br').alias('dh_processing_bronze_br'),
        expr('current_timestamp()-interval 3 hours').alias('dh_processing_br')
    )
).orderBy(col('match_id').asc())

#display(df_matches_clean)


In [0]:
df_matches_clean.write.mode('append').saveAsTable('api_football.silver.matches')

In [0]:
%sql
--TRUNCATE TABLE api_football.silver.matches --limpa a tabela (dados duplicados)